In [11]:
import os
import uuid
import pickle
from pathlib import Path
from typing import Any, List

from pydantic import ConfigDict
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
# from langchain_core.stores import InMemoryStore  # debug: não persiste entre sessões
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()

True

In [2]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=500)

In [3]:
pdf_dir = Path("./docs")
pdf_paths = list(pdf_dir.glob("*.pdf"))

documents = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

In [4]:
# Parent: chunks grandes — contexto rico para o LLM
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)

# Child: chunks pequenos — embeddings mais precisos para busca semântica
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
)

In [5]:
DOCSTORE_PATH = "./data/pdr_docstore.pkl"


class PickleDocStore:
    """Store persistente (pickle) com interface compatível com InMemoryStore."""

    def __init__(self, path: str):
        self._path = path
        self._store: dict = {}
        if Path(path).exists():
            with open(path, "rb") as f:
                self._store = pickle.load(f)

    def mset(self, pairs):
        self._store.update(pairs)
        with open(self._path, "wb") as f:
            pickle.dump(self._store, f)

    def mget(self, keys):
        return [self._store.get(k) for k in keys]

    def __len__(self):
        return len(self._store)


# docstore = InMemoryStore()  # debug: reseta ao reiniciar o kernel
docstore = PickleDocStore(DOCSTORE_PATH)

In [6]:
PDR_CHROMA_DIR = "./data/chroma_db_pdr"

vectorstore = Chroma(
    collection_name="pdr_children",
    embedding_function=embeddings_model,
    persist_directory=PDR_CHROMA_DIR,
)


def build_index(docs):
    parent_docs = parent_splitter.split_documents(docs)
    all_children = []
    for parent in parent_docs:
        parent_id = str(uuid.uuid4())
        docstore.mset([(parent_id, parent)])
        children = child_splitter.split_documents([parent])
        for child in children:
            child.metadata["parent_id"] = parent_id
        all_children.extend(children)
    vectorstore.add_documents(all_children)
    print(f"Indexed {len(parent_docs)} parents → {len(all_children)} child chunks")


if len(docstore) == 0:
    build_index(documents)
    print("Index built.")
else:
    print(f"Index already exists ({len(docstore)} parents). Skipping. "
          "Delete ./data/pdr_docstore.pkl and ./data/chroma_db_pdr/ to rebuild.")

Indexed 21 parents → 72 child chunks
Index built.


In [12]:
class PDRetriever(BaseRetriever):
    """Retriever PDR: busca child chunks, devolve os parent docs correspondentes."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    vectorstore: Any
    docstore: Any
    k: int = 3

    def _get_relevant_documents(self, query: str) -> List[Document]:
        child_hits = self.vectorstore.similarity_search(query, k=self.k * 4)
        seen, parents = set(), []
        for hit in child_hits:
            pid = hit.metadata.get("parent_id")
            if pid and pid not in seen:
                seen.add(pid)
                parent = self.docstore.mget([pid])[0]
                if parent:
                    parents.append(parent)
            if len(parents) == self.k:
                break
        return parents


retriever = PDRetriever(vectorstore=vectorstore, docstore=docstore)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Use the following context to answer the question. "
     "If the answer is not in the context, say you don't know. Be concise.\n\n"
     "{context}"),
    ("human", "{input}"),
])

rag_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))


def ask(question: str) -> str:
    return rag_chain.invoke({"input": question})["answer"]

In [14]:
user_question = input("Ask a question about the PDFs: ")
answer = ask(user_question)
print("Answer:", answer)

Answer: Pela história apresentada, o maior monstro conhecido que Vayne já matou foi a fera metamorfo, que ela conseguiu interromper durante sua transformação na toca da floresta ao leste de Demacia.
